# Location Analysis (new version)

This notebook processes and enriches location data from the Dehergne dataset to prepare it for geographic visualization and analysis.

**Workflow Stages:**

1.  **Data Extraction**:
    It gathers personal attributes related to locations (e.g., `nascimento`, `jesuita-entrada`, `morte`, `estadia`) into a unified DataFrame.

2.  **Date Inference**:
    Because many location attributes lack explicit dates, it infers them based on the chronological order of events in each person's biography (`ffill` method). It flags these inferred dates (`place.date_is_inferred`) for transparency.

3.  **Wikidata Enrichment**:
    It extracts Wikidata IDs from the source comments (e.g., `@wikidata:Q123`) to link historical place names to modern geodata.

4.  **Exports & Reports**:
    Generates several datasets in `../inferences/` for further use:
    *   `places_of_stay.xlsx`: Comprehensive list of stays with Wikidata IDs.
    *   `locations_names_places_dates_only.xlsx`: Simplified list for timeline visualization.
    *   `wikidata-references/locations_names_wikidata.csv`: Unique list of Wikidata IDs for the geocoding pipeline.
    *   `locations_names_no_wikidata.xlsx`: "To-do" list of places missing Wikidata links.
    *   `locations_totals.xlsx`: Frequency analysis of places (visit counts, date ranges).

5.  **Specific Analysis**:
    *   Filters missionaries by a specific place of entry (defaul: *Coimbra*).
    *   Exports individual markdown summaries associated with a specific place for review.



In [2]:
import os
import re
import pandas as pd
from timelink.notebooks import TimelinkNotebook
from timelink.pandas import entities_with_attribute
from timelink.kleio.utilities import format_timelink_date

pd.set_option("display.max_rows", 150)

tlnb = TimelinkNotebook()
db = tlnb.db
tlnb.print_info()

Timelink version: 1.1.33
Project name: dehergne
Project home: /Users/jrc/mhk-home/sources/dehergne
Database type: sqlite
Database name: dehergne
Kleio image: timelinkserver/kleio-server
Kleio server token: G9cKJ...
Kleio server URL: http://127.0.0.1:8088
Kleio server home: /Users/jrc/mhk-home/sources/dehergne
Kleio server container: affectionate_nobel
Kleio version requested: latest
Kleio server container status: running
Kleio server version: 12.9.592 (2026-03-19 23:14:02)
SQLite directory: /Users/jrc/mhk-home/sources/dehergne/database/sqlite
Database version: 6ccf1ef385a6
Call print_info(show_token=True) to show the Kleio Server token
Call print_info(show_password=True) to show the Postgres password
TimelinkNotebook(project_name=dehergne, project_home=/Users/jrc/mhk-home/sources/dehergne, db_type=sqlite, db_name=dehergne, kleio_image=timelinkserver/kleio-server, kleio_version=latest, postgres_image=postgres, postgres_version=latest)


## Attribute configuration
Attributes that store place names.

In [3]:
ATTRIBUTES_WITH_PLACE_NAMES = [
    'nascimento',
    'jesuita-entrada',
    'partida',
    'chegada',
    'estadia',
    'estadia-x',
    'jesuita-votos-local',
    'jesuita-ordenacao-padre',
    'morte',
]


## Visualization parameters

Columns to show in dataframe visualization

In [4]:

PREVIEW_COLS = [
    'place',
    'wikidata_id',
    'place.date_inferred.formatted',
    'place.date_inferred',
    'place.date_is_inferred',
    'name',
    'groupname',
    'place.type',
    'place.original',
    'place.line',
]

## Load place attributes and infer missing dates

We try to infer attributes with missing dates.
Sometimes the dates of some attribute are missing in the transcription, because in the original data they were not explicitly mentioned. 

But mostly the attributes are mentioned in cronological order so we can infer a relative date for attributes with missing dates (after date of the previous attribute).

This is the purpose of the following cell. It will create a new column called `place.date_inferred` which will recieve the an explicit date, if available or an inferred date. Another column called `place.date_is_inferred` with record if the date in `place.date_inferred` was inferred or not.

In [5]:
places_of_stay = entities_with_attribute(
    entity_type='person',
    show_elements=['name', 'groupname', 'the_line', 'the_order', 'extra_info'],
    the_type=[
        'jesuita-entrada',
        'partida',
        'chegada',
        'estadia',
        'estadia-x',
        'jesuita-votos-local',
        'jesuita-ordenacao-padre',
        'morte',
    ],
    column_name='place',
    db=db,
)

places_of_stay.place = places_of_stay.place.fillna('?')
places_of_stay = places_of_stay[places_of_stay.place != '?'].copy()

places_of_stay['id_col'] = places_of_stay.index
places_of_stay = places_of_stay.sort_values(by=['id_col', 'place.line'])

# Infer missing dates using the previous known date for each person
places_of_stay['place.date_inferred'] = places_of_stay['place.date'].replace('0', pd.NA)
places_of_stay['place.date_inferred'] = places_of_stay.groupby('id_col')['place.date_inferred'].ffill()
places_of_stay['place.date_is_inferred'] = places_of_stay['place.date_inferred'] != places_of_stay['place.date']
places_of_stay['place.date_inferred'] = places_of_stay['place.date_inferred'].fillna('')
places_of_stay['place.date_is_inferred'] = places_of_stay['place.date_inferred'] != places_of_stay['place.date']

places_of_stay.reset_index(inplace=True)
# if data_is_inferred set date_inferred to date_inferrred+">"
places_of_stay.loc[places_of_stay['place.date_is_inferred'], 'place.date_inferred'] = places_of_stay['place.date_inferred'] + '>'
places_of_stay['place.date_inferred'] = places_of_stay['place.date_inferred'].fillna('')
places_of_stay['place.date_inferred.formatted'] = places_of_stay['place.date_inferred'].apply(format_timelink_date)

places_of_stay.set_index('id_col', inplace=True)

places_of_stay.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6225 entries, deh-abraham-le-royer to simao-rodrigues-ref1
Data columns (total 24 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   id                             6225 non-null   object
 1   name                           6225 non-null   object
 2   groupname                      6225 non-null   object
 3   the_line                       6225 non-null   int64 
 4   the_order                      6225 non-null   int64 
 5   extra_info                     6225 non-null   object
 6   place.attr_id                  6225 non-null   object
 7   place.type                     6225 non-null   object
 8   place                          6225 non-null   object
 9   place.date                     6225 non-null   object
 10  place.line                     6225 non-null   int64 
 11  place.level                    6225 non-null   int64 
 12  place.groupname                6

## Extract wikidata IDs from extra info

This will extract the wikidata id from the comments of attribute values

In [6]:
import pandas as pd

from dehergne_util import extract_wikidata_id

places_of_stay['wikidata_id'] = places_of_stay['place.extra_info'].apply(extract_wikidata_id, if_missing=pd.NA)
places_of_stay.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6225 entries, deh-abraham-le-royer to simao-rodrigues-ref1
Data columns (total 25 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   id                             6225 non-null   object
 1   name                           6225 non-null   object
 2   groupname                      6225 non-null   object
 3   the_line                       6225 non-null   int64 
 4   the_order                      6225 non-null   int64 
 5   extra_info                     6225 non-null   object
 6   place.attr_id                  6225 non-null   object
 7   place.type                     6225 non-null   object
 8   place                          6225 non-null   object
 9   place.date                     6225 non-null   object
 10  place.line                     6225 non-null   int64 
 11  place.level                    6225 non-null   int64 
 12  place.groupname                6

In [7]:
def dataframe_overview(df, df_name="df", print_report=True):
    info_df = pd.DataFrame({
        '#': range(len(df.columns)),
        'Column': df.columns,
        'Non-Null Count': df.notna().sum().values,
        'Dtype': df.dtypes.astype(str).values,
    })

    info_df['Non-Null Count'] = info_df['Non-Null Count'].astype(str) + ' non-null'

    dtype_summary = df.dtypes.astype(str).value_counts().sort_index()
    dtype_text = ", ".join(f"{k}({v})" for k, v in dtype_summary.items())
    mem_mb = df.memory_usage(deep=True).sum() / (1024 ** 2)

    if print_report:
        if len(df.index) > 0:
            try:
                idx_min, idx_max = df.index.min(), df.index.max()
            except TypeError:
                idx_min, idx_max = df.index[0], df.index[-1]
            print(f"{df_name} -> Index: {len(df)} entries, {idx_min} to {idx_max}")
        else:
            print(f"{df_name} -> Index: 0 entries")

        print(f"Data columns (total {df.shape[1]} columns):")
        print(info_df.to_markdown(index=False))
        print(f"\ndtypes: {dtype_text}")
        print(f"memory usage: {mem_mb:.2f} MB")

    return info_df, dtype_summary, dtype_text, mem_mb


# Example with current dataframe
info_df, dtype_summary, dtype_text, mem_mb = dataframe_overview(places_of_stay, "places_of_stay")


places_of_stay -> Index: 6225 entries, deh-abraham-le-royer to simao-rodrigues-ref1
Data columns (total 25 columns):
|   # | Column                        | Non-Null Count   | Dtype   |
|----:|:------------------------------|:-----------------|:--------|
|   0 | id                            | 6225 non-null    | object  |
|   1 | name                          | 6225 non-null    | object  |
|   2 | groupname                     | 6225 non-null    | object  |
|   3 | the_line                      | 6225 non-null    | int64   |
|   4 | the_order                     | 6225 non-null    | int64   |
|   5 | extra_info                    | 6225 non-null    | object  |
|   6 | place.attr_id                 | 6225 non-null    | object  |
|   7 | place.type                    | 6225 non-null    | object  |
|   8 | place                         | 6225 non-null    | object  |
|   9 | place.date                    | 6225 non-null    | object  |
|  10 | place.line                    | 6225 non-null  

### Preview sample

Show place names of people in the Dehergne repertoire, avoiding the people "referred to" in the individual biographies, and so keeping just the missionaries names. These are the people registred under the group name "n". 

In [8]:
(
    places_of_stay[places_of_stay.groupname == 'n'][PREVIEW_COLS]
    .sort_values(by=['place', 'place.date_inferred', 'name'])
    .sample(50)
 )

,place,wikidata_id,place.date_inferred.formatted,place.date_inferred,place.date_is_inferred,name,groupname,place.type,place.original,place.line
id_col,,,,,,,,,,
deh-jean-francois-gerbillon,Molsheim,Q22697,1682-12-16,16821216,False,Jean-François Gerbillon,n,jesuita-ordenacao-padre,None,445
deh-pierre-foureau,Pequim,Q956,1735,1735,False,Pierre Foureau,n,estadia,None,1584
deh-alessandro-fillippucci,Macau,Q14773,1663,1663,False,Alessandro Fillippucci,n,estadia,None,1106
deh-andre-pereira,Macau,Q14773,1716-08-31,17160831,False,André Pereira,n,chegada,None,529
deh-miguel-vieira,Goa,Q1171,>1713,>1713,False,Miguel Vieira,n,estadia,None,1343
deh-aloys-kao,Paris (colégio Louis-le-Grand),Q90,>1751-01-07,>17510107,False,Aloys Kao,n,estadia,None,73
deh-giuseppe-candone,Macau,Q14773,1678->0,1678>,True,Giuseppe Candone,n,partida,None,234
deh-andre-pereira,Cantão,Q16572,1722,17220000,False,André Pereira,n,estadia,None,531
deh-jacques-niva-ni,Macau,Q14773,1638-10-26,16381026,False,Jacques Niva Ni,n,morte,None,110


## Save enriched datasets

In [9]:
(
    places_of_stay[PREVIEW_COLS]
    .sort_values(by=['place', 'place.date_inferred'])
    .to_excel('../inferences/locations_places_of_stay.xlsx', sheet_name='wikidata', index=True)
 )

(
    places_of_stay[['place', 'place.date_inferred.formatted', 'name', 'place.type', 'place.original']]
    .sort_values(by=['place', 'place.date_inferred.formatted'])
    .to_excel('../inferences/locations_names_places_dates_only.xlsx', sheet_name='locations', index=True)
 )

(
    places_of_stay['wikidata_id']
    .dropna()
    .drop_duplicates()
    .to_csv('../inferences/wikidata-references/locations_wikidata_ids.csv', index=False)
 )

## Locations missing wikidata IDs

In [10]:
places_without_wikidata = places_of_stay[
    places_of_stay['wikidata_id'].isna()
    & ~places_of_stay['place'].astype(str).str.startswith('[')
].copy()

(
    places_without_wikidata[PREVIEW_COLS]
    .sort_values(by=['place', 'place.date_inferred'])
    .sample(60)
 )

,place,wikidata_id,place.date_inferred.formatted,place.date_inferred,place.date_is_inferred,name,groupname,place.type,place.original,place.line
id_col,,,,,,,,,,
deh-antonio-posateri,Chang-si,<NA>,1691->0,1691>,True,Antonio Posateri,n,estadia,None,1836
deh-etienne-faber,"Siao-tchai de Han-tchong, Shensi",<NA>,1657-05-10,16570510,False,Etienne Faver,n,morte,None,52
deh-dionisio-ferreira,"Ko-li-tsem do Kiating, Kiangnan",<NA>,1753-12-08,17531208,False,Dionísio Ferreira,n,estadia,None,720
deh-simao-da-silveira,Aquilon,<NA>,1709-03-11,17090311>,True,Simão da Silveira,n,estadia,None,1443
deh-nicolas-marie-roy,"Lao chan oua, Hou-kouang",<NA>,1759-08-15,17590815,False,Nicolas-Marie Roy,n,jesuita-votos-local,None,1910
deh-manuel-da-motta,"U cham fu, Hou-kouang",<NA>,1767-12-13,17671213,False,Manuel da Motta,n,jesuita-votos-local,None,2336
deh-joao-barradas,Rio Kan,<NA>,1594,15940000,False,João Barradas,n,morte,None,356
deh-manswet-skokowski,Mscislaviae,<NA>,1798-02-08,17980208,False,Manswet Skokowvski,n,morte,None,1751
deh-louis-joseph-le-febure,San kiao hou kia,<NA>,1741-03-19,17410319,False,Louis-Joseph Le Febure,n,jesuita-votos-local,None,884


In [11]:
(
    places_without_wikidata[PREVIEW_COLS]
    .sort_values(by=['place', 'place.date_inferred'])
    .to_excel('../inferences/wikidata-references/locations_names_no_wikidata.xlsx', sheet_name='no_wikidata', index=True)
 )


## Important

For this to work make sure that notebook `wikidata-linked-data.ipynb` has been run. It will read the wikidate exported above to `../inferences/wikidata-references/` and fetch the information of each location to locations_wikidata_info.xlsx which is used in the next cell to provide extra information about each location

## Frequency of places

### Merge places with wikidata information


In [12]:
# get information about wikidata locations. See notebooks/wikidata-linked-data.ipynb for how this file was created
locations_wikidata_info = pd.read_excel(
    "../inferences/wikidata-references/locations_wikidata_info.xlsx"
)

# Normalize join keys
locations_wikidata_info["wikidata_id"] = locations_wikidata_info["wikidata_id"].astype("string").str.strip()
places_of_stay["wikidata_id"] = places_of_stay["wikidata_id"].astype("string").str.strip()

# Ensure many-to-one join (one row per wikidata_id on lookup table)
locations_wikidata_info = locations_wikidata_info.drop_duplicates(subset=["wikidata_id"], keep="first")

# Left join: keep all rows from places_of_stay and copy columns from locations_wikidata_info
places_of_stay = (
    places_of_stay.reset_index()
    .merge(
        locations_wikidata_info,
        how="left",
        left_on="wikidata_id",
        right_on="wikidata_id",
        suffixes=("", "_wikidata"),
    )
    .set_index("id_col")
)

places_of_stay.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6225 entries, deh-abraham-le-royer to simao-rodrigues-ref1
Data columns (total 41 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   id                              6225 non-null   object 
 1   name                            6225 non-null   object 
 2   groupname                       6225 non-null   object 
 3   the_line                        6225 non-null   int64  
 4   the_order                       6225 non-null   int64  
 5   extra_info                      6225 non-null   object 
 6   place.attr_id                   6225 non-null   object 
 7   place.type                      6225 non-null   object 
 8   place                           6225 non-null   object 
 9   place.date                      6225 non-null   object 
 10  place.line                      6225 non-null   int64  
 11  place.level                     6225 non-null   int64  
 12  plac

### Show sample records



In [13]:
cols = [ 'name', 'place', 'place.type','portuguese_label','chinese_label','label','country_label','wikidata_id','english_description', 'administrative_entity_id', 'administrative_entity_label_en']
places_of_stay[cols].sample(10)

,name,place,place.type,portuguese_label,chinese_label,label,country_label,wikidata_id,english_description,administrative_entity_id,administrative_entity_label_en
id_col,,,,,,,,,,,
deh-simon-xavier-a-cunha-wou-li,Simon-Xavier a Cunha Wou Li,Shanghai,estadia,Xangai,上海市,Shanghai / 上海市,People's Republic of China,Q8686,provincial-level municipality in China,Q148,People's Republic of China
deh-etienne-joseph-le-couteulx,Etienne-Joseph Le Couteulx,Hankow,estadia,NaN,汉口,Hankou / 汉口,Republic of China,Q1208250,"former town, now part of Wuhan",Q13426199,Republic of China
deh-jean-francois-foucquet,Jean-François Foucquet,Port-Louis,estadia,Port-Louis,路易港,Port-Louis / 路易港,France,Q128223,"commune in Morbihan, France",Q178875,canton of Port-Louis
deh-antoine-thomas,Antoine Thomas,Namur,estadia,Namur,那慕爾,Namur / 那慕爾,Belgium,Q134121,"capital of Wallonia, Belgium",Q93782,Arrondissement of Namur
deh-afonso-aires,Afonso Aires,"[No mar, depois do Cabo da Boa Esperança]",morte,Cabo da Boa Esperança,好望角,Cape of Good Hope / 好望角,South Africa,Q4092,"headland of Cape Peninsula, South Africa",Q127167,Western Cape
deh-jean-francois-pelisson,Jean-François Pélisson,Toulouse,morte,Toulouse,圖盧茲,Toulouse / 圖盧茲,France,Q7880,"French commune and city in Haute-Garonne, Occi...",Q12538,Haute-Garonne
deh-houang-francisco-martins,Houang Francisco Martins,Shiuchow,estadia,NaN,曲江区,Qujiang District / 曲江区,People's Republic of China,Q1346617,"district in Guangdong, People's Republic of China",Q59107,Shaoguan
deh-pedro-de-alcacova,Pedro de Alcáçova,Japão,estadia,Japão,日本,Japan / 日本,Japan,Q17,island country in East Asia,NaN,NaN
deh-joao-de-borgia-kouo,João de Borgia Kouo,Macau,morte,Macau,澳門,NaN,People's Republic of China,Q14773,city and special administrative region of China,Q148,People's Republic of China


In [14]:
df_totals = (
    places_of_stay.reset_index()
    .groupby(['portuguese_label','chinese_label','english_label'])
    .agg(id_unique=('id_col', 'nunique'), first_date=('place.date_inferred', 'min'), last_date=('place.date_inferred', 'max'))
    .sort_values('id_unique', ascending=False)
)

df_totals.head(100)

,,,id_unique,first_date,last_date
portuguese_label,chinese_label,english_label,,,
Macau,澳門,Macau,476,1561,>1652
Pequim,北京市,Beijing,253,00,>17410115
Goa,果阿邦,Goa,223,00,>1713
Cantão,广州市,Guangzhou,204,1555,>16590123
China,中华人民共和国,People's Republic of China,184,15540000,>
Lisboa,里斯本,Lisbon,150,1546,>
Roma,罗马,Rome,113,15480311,>16851112
Coimbra,科英布拉,Coimbra,79,15420000,<16230324
Paris,巴黎,Paris,72,1527,>17510107


In [15]:
df_totals.to_excel('../inferences/locations_totals.xlsx', sheet_name='totals', index=True)

### By country or region

In [16]:
df_countries = (
    places_of_stay.reset_index()
    .groupby(['country_label'])
    .agg(id_unique=('id_col', 'nunique'), first_date=('place.date_inferred', 'min'), last_date=('place.date_inferred', 'max'))
    .sort_values('id_unique', ascending=False)
)

df_countries

,id_unique,first_date,last_date
country_label,,,
People's Republic of China,887,00,>17410115
India,322,00,>1713
Portugal,264,15420000,>1707:<17160314
France,238,1527,>17510107
Italy,195,15370624,>16851112
Vietnam,94,16190000,>
Qing dynasty,91,1619,>1710
Japan,78,15490815,>
Thailand,56,1568,>


In [17]:
df_countries.to_excel('../inferences/locations_countries.xlsx', sheet_name='countries', index=True)

## Analysis of name variations



In [18]:
df_variations = (
    places_of_stay.reset_index()
    .groupby(['country_label','portuguese_label','chinese_label','english_label','wikidata_id','place'])
    .agg(id_unique=('id_col', 'nunique'), first_date=('place.date_inferred', 'min'), last_date=('place.date_inferred', 'max'))
)
df_variations.head(100)

id_unique  \
country_label      portuguese_label           chinese_label english_label              wikidata_id place                                          
Abbasid Caliphate  Coração                    呼罗珊           Khorasan                   Q2352322    Khorasan                                   1   
Afghanistan        Afeganistão                阿富汗           Afghanistan                Q889        Afeganistão                                1   
Angola             Angola                     安哥拉           Angola                     Q916        Angola                                     1   
Argentina          Argentina                  阿根廷           Argentina                  Q414        Missão de Santa Ana, dos Guaranis          1   
                   Buenos Aires               布宜諾斯艾利斯       Buenos Aires               Q1486       Buenos Aires                               1   
Armenia            Arménia                    亞美尼亞          Armenia                    Q399        Arménia                                    6   
                                                                                                   Arménia (residência de Chamakii)           1   
                   Erevã                      葉里溫           Yerevan                    Q1953       Yerevan                                    2   
Austria            Graz                       格拉茨           Graz                       Q13298      Graz                                       2   
                   Krems an der Donau         多瑙河畔克雷姆斯      Krems                      Q131266     Krems                                      1   
                   Leoben                     萊奧本           Leoben                     Q485294     Leoben                                     2   
                   Viena                      維也納           Vienna                     Q1741       Viena                                     14   
Azerbaijan         Shamakhi                   沙馬基           Shamakhi                   Q388224     Shamakhi                                   1   
Belarus            Bielorrússia               白俄羅斯          Belarus                    Q184        Rússia Branca                              4   
                   Minsk                      明斯克           Minsk                      Q2280       Minsk                                      1   
                   Polatsk                    波洛茨克          Polatsk                    Q200797     Polotsk                                    5   
Belgium            Antuérpia                  安特卫普          Antwerp                    Q12892      Antuérpia                                  6   
                   Bruges                     布鲁日           Bruges                     Q12994      Bruges                                     1   
                   Bruxelas                   布鲁塞尔          City of Brussels           Q239        Bruxelas                                   5   
                   Courtrai                   科特赖克          Kortrijk                   Q12995      Courtrai                                   2   
                   Liège                      列日            Liège                      Q3992       Liège                                      1   
                                                                                                   Liége                                      1   
                   Lovaina                    鲁汶            Leuven                     Q118958     Leuven                                     6   
                   Mechelen                   梅赫伦           Mechelen                   Q162022     Mechelen                                  10   
                   Namur                      那慕爾           Namur                      Q134121     Namur                                      1   
                   Oostende                   奥斯坦德          Ostend                     Q12996      Ostende                                    1   
                  

In [19]:
df_variations.reset_index().to_excel('../inferences/wikidata-references/locations_variations.xlsx', sheet_name='variations', index=True)

## Produce a Gazeeter 

Same data but alphabetical order of place, place in the first column, followed by wikidata_id, first_date, last_date, unique_id, and the labels

In [20]:
df_variations.reset_index()[['place', 'wikidata_id', 'country_label', 'portuguese_label', 'chinese_label', 'english_label', 'id_unique','first_date','last_date']].sort_values(by=['place']).to_excel('../inferences/locations_geonames.xlsx', index=False)

### Place names variations for a specific country

In [21]:
unique_countries = (
    df_variations.index.get_level_values("country_label")
    .dropna()
    .unique()
    .tolist()
)

sorted(unique_countries)

['Abbasid Caliphate',
 'Afghanistan',
 'Angola',
 'Argentina',
 'Armenia',
 'Austria',
 'Azerbaijan',
 'Belarus',
 'Belgium',
 'Bolivia',
 'Brazil',
 'British Raj',
 'Cambodia',
 'Canada',
 'Champa',
 'Chile',
 'Croatia',
 'Cuba',
 'Czech Republic',
 'Dominican Republic',
 'Duchy of Lorraine',
 'Duchy of Moscow',
 'Dutch East Indies',
 'Ecuador',
 'Egypt',
 'Ethiopia',
 'France',
 'Germany',
 'Greece',
 'Guinea',
 'Hungary',
 'India',
 'Indonesia',
 'Iran',
 'Italy',
 'Japan',
 'Kazakhstan',
 'Kingdom of Hungary',
 'Kingdom of Italy',
 'Kingdom of Poland',
 'Kingdom of Romania',
 'Kingdom of the Netherlands',
 'Kyrgyzstan',
 'Lebanon',
 'Lithuania',
 'Malaysia',
 'Mexico',
 'Morocco',
 'Mozambique',
 'Mughal Empire',
 'Myanmar',
 'Nepal',
 'Netherlands',
 'Norway',
 'Pakistan',
 'Paraguay',
 "People's Republic of China",
 'Peru',
 'Philippines',
 'Poland',
 'Portugal',
 'Portuguese Empire',
 'Qing dynasty',
 'Roman Empire',
 'Romania',
 'Russia',
 'Slovakia',
 'South Africa',
 'Spain',

In [22]:
country_of_interest = "People's Republic of China"
country_of_interest_df = df_variations[df_variations.index.get_level_values('country_label') == country_of_interest]
country_of_interest_df.head(100)

id_unique  \
country_label              portuguese_label        chinese_label english_label              wikidata_id place                                                         
People's Republic of China Anlu                    安陆市           Anlu                       Q557934     Anlo (Ngan-lo), Hupeh                                     1   
                                                                                                        Anlu (Ngan-lou), Hupei                                    1   
                           Anqing                  安庆市           Anqing                     Q360374     Anking (Ngan-k'ing fou)                                   1   
                           Cantão                  广东省           Guangdong                  Q15175      Kouang-tong                                               2   
                                                                                                        Kwangtung                                                16   
                                                                                                        Kwangtung, China                                          1   
                                                   广州市           Guangzhou                  Q16572      Cantão                                                  197   
                                                                                                        Cantão (enfermaria dos franciscanos, extramuros)          1   
                                                                                                        Cantão (feira)                                            3   
                                                                                                        Cantão (igreja Ti-lo-pu)                                  1   
                                                                                                        Cantão (igreja de Ty-lo-po)                               1   
                                                                                                        Cantão (in sancello resid. Gallicae S.J.)                 1   
                                                                                                        Cantão (missão)                                           1   
                                                                                                        Cantão (porto, navio Amphitrite)                          1   
                                                                                                        Cantão (residência francesa)                              1   
                                                                                                        Cantão (residência)                                       1   
                                                                                                        Guangzhou                                                 1   
                                                                                                        [Costa de Cantão]                                         1   
                           Cemitério de Chala      滕公栅栏          Zhalan Cemetery            Q4578286    Chala                                                     1   
                           Changsha                长沙市           Changsha                   Q174091     Changsha                                                  1   
                           Changzhou               常州市           Changzhou                  Q57970      Changchow, China                                          1   
                           Chengde                 承德市           Chengde                    Q58675      Jehol                                                     3   
                                                                                                        Jéhol                                     

### Export country of interest places

In [23]:
filename = f"../inferences/locations_{country_of_interest}_name_variations.xlsx"

country_of_interest_df.to_excel(filename, sheet_name='variations', index=True)

## Who was at a specific place

In [24]:
place_of_interest = 'Coimbra'

In [25]:
cols_place_detail = ['place', 'wikidata_id', 'name', 'place.type', 'place.date_inferred', 'place.obs']
(
    places_of_stay[places_of_stay.place == place_of_interest][cols_place_detail]
    .sort_values(by=['place.date_inferred'])
    .head(10)
)

,place,wikidata_id,name,place.type,place.date_inferred,place.obs
id_col,,,,,,
deh-pedro-de-alcacova,Coimbra,Q45412,Pedro de Alcáçova,jesuita-entrada,15420000,saiu a primeira vez
deh-belchior-nunes-barreto,Coimbra,Q45412,Belchior Nunes Barreto,jesuita-entrada,15430311,
deh-belchior-miguel-carneiro-leitao,Coimbra,Q45412,Belchior Miguel Carneiro Leitão,jesuita-entrada,15430425,
deh-francisco-perez,Coimbra,Q45412,Francisco Pérez,jesuita-entrada,15440125,
deh-tiburcio-de-quadros,Coimbra,Q45412,Tibúrcio de Quadros,jesuita-entrada,15440418,
deh-goncalo-alvares,Coimbra,Q45412,Gonçalo Álvares,jesuita-entrada,15490101,
deh-cristovao-da-costa,Coimbra,Q45412,Cristóvão da Costa,jesuita-entrada,15500103,
deh-joao-rodrigues-de-sainan,Coimbra,Q45412,João Rodrigues de Sainan,jesuita-entrada,1555,
deh-pedro-martins,Coimbra,Q45412,Pedro Martins,jesuita-entrada,15560525,


In [26]:
wikidata_id_of_interest='Q1207099'
cols_place_detail = ['place', 'wikidata_id', 'name', 'place.type', 'place.date_inferred', 'place.obs']
(
    places_of_stay[places_of_stay.wikidata_id == wikidata_id_of_interest][cols_place_detail]
    .sort_values(by=['place.date_inferred'])
    .head(100)
)

,place,wikidata_id,name,place.type,place.date_inferred,place.obs
id_col,,,,,,
deh-jean-valat,Lin-ts'ing,Q1207099,Jean Valat,estadia,1685,
deh-jean-charles-etienne-froissard-de-broissia,"[No mar, para lá de Lin-tsing (Lin-ts'ing, Sha...",Q1207099,Jean-Charles Etienne Froissard de Broissia,morte,17040918,
deh-jose-simoes,Lintsing,Q1207099,José Simões,estadia,17070600,
deh-pierre-tchen,"Linkiang, Kiangsi",Q1207099,Pierre T'chen,estadia,17430000,


### Export markdown stubs for persons at that place

In [27]:
markdown_directory = f"../inferences/{place_of_interest}/markdown"
os.makedirs(markdown_directory, exist_ok=True)

for person_id, name in places_of_stay[places_of_stay.place == place_of_interest][['name']].sort_values('name').itertuples(index=True):
    with tlnb.db.session() as session:
        person = tlnb.db.get_person(person_id, session=session)
        session.add(person)
        kleio_text = person.to_kleio()

    note = f"""## {name}
^[In Edit mode, select title (including the ##), right click and "Extract selection"]

---
### {name} @ timelink
```
{kleio_text}
```

---
Select title, right click on selected title, choose extract note
"""
    
    with open(f"{markdown_directory}/{person_id}.md", "w", encoding="utf-8") as handle:
        handle.write(note)

## Recover missing Wikidata IDs from variations

Try to find candidate Wikidata IDs for locations missing them, using the variations file that contains variations of names for known Wikidata IDs.

In [28]:
import pandas as pd
import os
from difflib import SequenceMatcher

# Define file paths
no_wikidata_path = '../inferences/wikidata-references/locations_names_no_wikidata.xlsx'
variations_path = '../inferences/wikidata-references/locations_variations.xlsx'
output_path = '../inferences/wikidata-references/locations_names_no_wikidata_with_candidates.xlsx'

# Check if files exist
if not os.path.exists(no_wikidata_path):
    print(f"File not found: {no_wikidata_path}")
    # Try alternative path based on notebook logic
    no_wikidata_path = '../inferences/locations_names_no_wikidata.xlsx'
    print(f"Trying: {no_wikidata_path}")

if not os.path.exists(variations_path):
    print(f"File not found: {variations_path}")
    # Try alternative path
    variations_path = '../inferences/locations_variations.xlsx'
    print(f"Trying: {variations_path}")

try:
    df_missing = pd.read_excel(no_wikidata_path)
    df_variations = pd.read_excel(variations_path)

    print(f"Loaded {len(df_missing)} rows with missing Wikidata IDs.")
    print(f"Loaded {len(df_variations)} known location variations.")

    # 1. Basic Cleaning
    df_missing['place_clean'] = df_missing['place'].astype(str).str.strip()
    df_variations['place_clean'] = df_variations['place'].astype(str).str.strip()

    # 2. Prepare candidates
    # We only care about variations that actually have a Wikidata ID
    valid_variations = df_variations.dropna(subset=['wikidata_id'])[['place_clean', 'wikidata_id']].drop_duplicates()
    candidate_names = valid_variations['place_clean'].tolist()
    candidate_ids = valid_variations['wikidata_id'].tolist()
    
    # Create an exact lookup first for O(1) matching
    exact_lookup = dict(zip(candidate_names, candidate_ids))

    # 3. Fuzzy matching function
    def find_best_match(query, choices, ids, cutoff=0.6):
        """
        Find best string match using difflib.SequenceMatcher.
        Returns (id, match_name, score)
        """
        # Exact match optimization
        if query in exact_lookup:
            return exact_lookup[query], query, 1.0

        best_match = None
        best_id = None
        best_ratio = 0.0
        
        for choice, wid in zip(choices, ids):
            # SequenceMatcher gives a ratio between 0 and 1
            ratio = SequenceMatcher(None, query, choice).ratio()
            if ratio > best_ratio:
                best_ratio = ratio
                best_match = choice
                best_id = wid
        
        if best_ratio >= cutoff:
            return best_id, best_match, best_ratio
        return None, None, 0.0

    print("Finding matches (this might take a while)...")
    
    # 4. Apply matching
    # Work on unique names to avoid re-calcuating the same name multiple times
    unique_missing = df_missing['place_clean'].unique()
    print(f"Processing {len(unique_missing)} unique missing place names...")
    
    results = []
    for i, place in enumerate(unique_missing):
        wid, match_name, score = find_best_match(place, candidate_names, candidate_ids, cutoff=0.65)
        results.append({
            'place_clean': place,
            'candidate_id': wid,
            'candidate_name': match_name,
            'match_score': score
        })
        if (i + 1) % 50 == 0:
            print(f"Processed {i + 1}/{len(unique_missing)}...")
    
    df_results = pd.DataFrame(results)
    
    # 5. Merge results back
    df_final = df_missing.merge(df_results, on='place_clean', how='left')

    # 6. Report and Save
    found_matches = df_final[df_final['candidate_id'].notna()]
    print(f"Found {len(found_matches)} candidates out of {len(df_missing)} rows.")

    if len(found_matches) > 0:
        print("\nTop matches (by score):")
        cols_to_show = ['place', 'candidate_name', 'match_score', 'candidate_id']
        print(found_matches[cols_to_show].drop_duplicates(subset=['place']).sort_values('match_score', ascending=False).head(15))

    df_final.drop(columns=['place_clean'], inplace=True, errors='ignore')
    df_final.to_excel(output_path, index=False)
    print(f"\nSaved {len(df_final)} rows to {output_path}")

except Exception as e:
    print(f"Error: {e}")
    import traceback
    traceback.print_exc()

Loaded 60 rows with missing Wikidata IDs.
Loaded 843 known location variations.
Finding matches (this might take a while)...
Processing 54 unique missing place names...
Processed 50/54...
Found 11 candidates out of 60 rows.

Top matches (by score):
                              place                candidate_name  \
4                          Chang-si                       Chan-si   
15                 Ké Diet, Tonquim                 Hien, Tonquim   
20                  Làngòi, Tonquim                 Hien, Tonquim   
27                          Oriente                       Lorient   
13  Ko-li-tsem do Kiating, Kiangnan  Kiatim (Kiating) do Kiangnan   
24                Monzei, Filipinas                     Filipinas   
38                    Sian tan hien        Suian, Souei-ngan hien   
6               Chayüankow, Hukwang             Hengchow, Hukwang   
25                          Moulins                         Minsk   
33          Pé-tsiuen-chan, Hukwang             Hengchow, Huk

In [29]:
df_final.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 14 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   id_col                         60 non-null     object 
 1   place                          60 non-null     object 
 2   wikidata_id                    0 non-null      float64
 3   place.date_inferred.formatted  60 non-null     object 
 4   place.date_inferred            60 non-null     object 
 5   place.date_is_inferred         60 non-null     bool   
 6   name                           60 non-null     object 
 7   groupname                      60 non-null     object 
 8   place.type                     60 non-null     object 
 9   place.original                 1 non-null      object 
 10  place.line                     60 non-null     int64  
 11  candidate_id                   11 non-null     object 
 12  candidate_name                 11 non-null     objec

We generate a document with the biographies of all the peopl with references to non located places